Theorem 2.6 - any n lines in the plane (not necessarily general position) split it into
regions that can be 2-colored so adjacent regions differ.

input: `lines` as (a, b, c) triples meaning ax+by+c=0, and a query `point` (x, y) not on any line

output: 0/1 color for the point's region. Construction taken straight from the proof: color
= parity of how many lines the point is on the positive side of. Crossing one line flips
exactly one term of that sum, so adjacent regions always disagree - that's what the proof's
flip-the-colors-on-one-side step is doing.

In [ ]:
import random


def side(line, point):
    a, b, c = line
    x, y = point
    return 1 if a * x + b * y + c > 0 else 0


def region_color(point, lines):
    return sum(side(l, point) for l in lines) % 2


def verify_two_coloring(lines, trials=200, eps=1e-6, seed=0):
    '''
    For random points sitting on one line and pushed eps to either side, the two
    pushed points must get different colors - that is exactly the adjacency the
    theorem claims. Trials landing too close to a second line are skipped since
    then the "only one line separates them" assumption no longer holds.
    '''
    rng = random.Random(seed)
    checked = 0
    for _ in range(trials):
        i = rng.randrange(len(lines))
        a, b, c = lines[i]
        if b != 0:
            x = rng.uniform(-5, 5)
            y = -(a * x + c) / b
        else:
            y = rng.uniform(-5, 5)
            x = -(b * y + c) / a
        norm = (a ** 2 + b ** 2) ** 0.5
        nx, ny = a / norm, b / norm
        p1 = (x + eps * nx, y + eps * ny)
        p2 = (x - eps * nx, y - eps * ny)
        if any(abs(l[0] * x + l[1] * y + l[2]) < 1e-3 for j, l in enumerate(lines) if j != i):
            continue
        assert region_color(p1, lines) != region_color(p2, lines)
        checked += 1
    return checked

In [ ]:
lines = [
    (1, 0, 0),    # x = 0
    (0, 1, 0),    # y = 0
    (1, 1, -3),   # x + y = 3
    (1, -1, 1),   # x - y = -1
]

print("color(2, 2) =", region_color((2, 2), lines))
print("color(-1, -1) =", region_color((-1, -1), lines))

checked = verify_two_coloring(lines)
print(f"checked {checked} line crossings, every one flips color as the theorem requires")